In [ ]:
!pip install wfdb

In [ ]:
import wfdb

wfdb.dl_database(
    "picsdb",
    dl_dir="picsdb_dataset"
)

Generating record list for: infant10_ecg
Generating record list for: infant10_resp
Generating record list for: infant1_ecg
Generating record list for: infant1_resp
Generating record list for: infant2_ecg
Generating record list for: infant2_resp
Generating record list for: infant3_ecg
Generating record list for: infant3_resp
Generating record list for: infant4_ecg
Generating record list for: infant4_resp
Generating record list for: infant5_ecg
Generating record list for: infant5_resp
Generating record list for: infant6_ecg
Generating record list for: infant6_resp
Generating record list for: infant7_ecg
Generating record list for: infant7_resp
Generating record list for: infant8_ecg
Generating record list for: infant8_resp
Generating record list for: infant9_ecg
Generating record list for: infant9_resp
Generating list of all files for: infant10_ecg
Generating list of all files for: infant10_resp
Generating list of all files for: infant1_ecg
Generating list of all files for: infant1_resp


KeyboardInterrupt: 

In [ ]:
import os

os.listdir("picsdb_dataset")[:10]

In [ ]:
import os
import wfdb
import numpy as np
from scipy.signal import find_peaks

all_heart_rates = []
all_labels = []

dataset_path = "picsdb_dataset"

for file in os.listdir(dataset_path):

    if file.endswith("_ecg.dat"):

        record_name = file.replace(".dat","")

        record = wfdb.rdrecord(os.path.join(dataset_path, record_name))

        ecg_signal = record.p_signal[:,0]

        peaks, _ = find_peaks(ecg_signal, distance=50)

        rr_intervals = np.diff(peaks)

        heart_rate = 60 / (rr_intervals / record.fs)

        labels = heart_rate < 100

        all_heart_rates.extend(heart_rate)
        all_labels.extend(labels)

X = np.array(all_heart_rates).reshape(-1,1)
y = np.array(all_labels).astype(int)

print("Total samples:", len(y))
print("Bradycardia samples:", np.sum(y))

In [ ]:
features = []
labels = []

dataset_path = "picsdb_dataset"

for file in os.listdir(dataset_path):

    if file.endswith("_ecg.dat"):

        record_name = file.replace(".dat","")

        record = wfdb.rdrecord(os.path.join(dataset_path, record_name))

        ecg_signal = record.p_signal[:,0]

        peaks, _ = find_peaks(ecg_signal, distance=50)

        rr_intervals = np.diff(peaks)

        heart_rate = 60 / (rr_intervals / record.fs)

        for i in range(len(rr_intervals)-5):

            window = rr_intervals[i:i+5]

            mean_rr = np.mean(window)
            std_rr = np.std(window)
            min_rr = np.min(window)
            max_rr = np.max(window)

            features.append([mean_rr, std_rr, min_rr, max_rr])

            labels.append(heart_rate[i] < 100)

X = np.array(features)
y = np.array(labels).astype(int)

print("Dataset shape:", X.shape)
print("Bradycardia samples:", np.sum(y))

In [ ]:
print("Total samples:", len(y))
print("Bradycardia samples:", np.sum(y))
print("Normal samples:", len(y) - np.sum(y))

In [ ]:
import numpy as np

# indices of both classes
brady_idx = np.where(y == 1)[0]
normal_idx = np.where(y == 0)[0]

# randomly select equal number of normal samples
normal_sample = np.random.choice(normal_idx, size=len(brady_idx), replace=False)

# combine indices
balanced_idx = np.concatenate([brady_idx, normal_sample])

# create balanced dataset
X_balanced = X[balanced_idx]
y_balanced = y[balanced_idx]

print("Balanced dataset size:", len(y_balanced))
print("Bradycardia samples:", np.sum(y_balanced))
print("Normal samples:", len(y_balanced) - np.sum(y_balanced))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_balanced,
    y_balanced,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(y_train))
print("Testing samples:", len(y_test))

In [ ]:
#Train SVM Model
from sklearn.svm import SVC

svm_model = SVC(kernel='rbf')

svm_model.fit(X_train, y_train)

In [ ]:
#Evaluate SVM
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

svm_pred = svm_model.predict(X_test)

svm_accuracy = accuracy_score(y_test, svm_pred)
svm_precision = precision_score(y_test, svm_pred)
svm_recall = recall_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred)

svm_results = pd.DataFrame({
    "Model": ["SVM"],
    "Accuracy": [round(svm_accuracy,4)],
    "Precision": [round(svm_precision,4)],
    "Recall": [round(svm_recall,4)],
    "F1 Score": [round(svm_f1,4)]
})

svm_results

In [ ]:
#Train Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

In [ ]:
#Evaluate Random Forest
rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)

rf_results = pd.DataFrame({
    "Model": ["Random Forest"],
    "Accuracy": [round(rf_accuracy,4)],
    "Precision": [round(rf_precision,4)],
    "Recall": [round(rf_recall,4)],
    "F1 Score": [round(rf_f1,4)]
})

rf_results

In [ ]:
#Reshape Data for Deep Learning
import numpy as np

X_train_dl = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_dl = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(X_train_dl.shape)
print(X_test_dl.shape)

In [ ]:
#CNN Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense

cnn_model = Sequential([
    Conv1D(filters=32, kernel_size=2, activation='relu', input_shape=(X_train_dl.shape[1],1)),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

cnn_model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

cnn_model.fit(X_train_dl, y_train, epochs=10, batch_size=16, verbose=1)

In [ ]:
#CNN Evaluation
cnn_pred = (cnn_model.predict(X_test_dl) > 0.5).astype(int)

cnn_accuracy = accuracy_score(y_test, cnn_pred)
cnn_precision = precision_score(y_test, cnn_pred)
cnn_recall = recall_score(y_test, cnn_pred)
cnn_f1 = f1_score(y_test, cnn_pred)

cnn_results = pd.DataFrame({
    "Model": ["CNN"],
    "Accuracy": [round(cnn_accuracy,4)],
    "Precision": [round(cnn_precision,4)],
    "Recall": [round(cnn_recall,4)],
    "F1 Score": [round(cnn_f1,4)]
})

cnn_results

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, cnn_pred))

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf_model, X_balanced, y_balanced, cv=5)

print("Cross-validation accuracy:", scores)
print("Mean accuracy:", scores.mean())

In [ ]:
import numpy as np

X_train_dl = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_dl = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("Train shape:", X_train_dl.shape)
print("Test shape:", X_test_dl.shape)

In [ ]:
#Build LSTM Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

lstm_model = Sequential([
    LSTM(32, input_shape=(X_train_dl.shape[1],1)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

lstm_model.fit(X_train_dl, y_train, epochs=10, batch_size=16, verbose=1)

In [ ]:
#Evaluate LSTM
lstm_pred = (lstm_model.predict(X_test_dl) > 0.5).astype(int)

lstm_accuracy = accuracy_score(y_test, lstm_pred)
lstm_precision = precision_score(y_test, lstm_pred)
lstm_recall = recall_score(y_test, lstm_pred)
lstm_f1 = f1_score(y_test, lstm_pred)

lstm_results = pd.DataFrame({
    "Model": ["LSTM"],
    "Accuracy": [round(lstm_accuracy,4)],
    "Precision": [round(lstm_precision,4)],
    "Recall": [round(lstm_recall,4)],
    "F1 Score": [round(lstm_f1,4)]
})

lstm_results

In [ ]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

cnn_bilstm_model = Sequential([

    Conv1D(filters=32, kernel_size=2, activation='relu',
           input_shape=(X_train_dl.shape[1],1)),

    MaxPooling1D(pool_size=2),

    Bidirectional(LSTM(32)),

    Dropout(0.2),

    Dense(16, activation='relu'),

    Dense(1, activation='sigmoid')
])

cnn_bilstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

cnn_bilstm_model.fit(X_train_dl, y_train, epochs=10, batch_size=16, verbose=1)

In [ ]:
cnn_bilstm_pred = (cnn_bilstm_model.predict(X_test_dl) > 0.5).astype(int)

neo_accuracy = accuracy_score(y_test, cnn_bilstm_pred)
neo_precision = precision_score(y_test, cnn_bilstm_pred)
neo_recall = recall_score(y_test, cnn_bilstm_pred)
neo_f1 = f1_score(y_test, cnn_bilstm_pred)

neo_results = pd.DataFrame({
    "Model": ["CNN + BiLSTM (NeoPredict)"],
    "Accuracy": [round(neo_accuracy,4)],
    "Precision": [round(neo_precision,4)],
    "Recall": [round(neo_recall,4)],
    "F1 Score": [round(neo_f1,4)]
})

neo_results

In [ ]:
import matplotlib.pyplot as plt

table_data = [
    ["SVM",
     round(svm_accuracy*100,1),
     round(svm_precision*100,1),
     round(svm_recall*100,1),
     round(svm_f1*100,1)],

    ["Random Forest",
     round(rf_accuracy*100,1),
     round(rf_precision*100,1),
     round(rf_recall*100,1),
     round(rf_f1*100,1)],

    ["CNN-LSTM",
     round(cnn_accuracy*100,1),
     round(cnn_precision*100,1),
     round(cnn_recall*100,1),
     round(cnn_f1*100,1)],

    ["NeoPredict(Proposed)",
     round(neo_accuracy*100,1),
     round(neo_precision*100,1),
     round(neo_recall*100,1),
     round(neo_f1*100,1)]
]

columns = ["Model", "Accuracy (%)", "Precision (%)", "Recall (%)", "F1-Score (%)"]

fig, ax = plt.subplots(figsize=(10,3))
ax.axis('off')

table = ax.table(
    cellText=table_data,
    colLabels=columns,
    cellLoc='center',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(2,2)

plt.title("Table_Model_Performance_Comparison", fontsize=14, fontweight='bold')

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

plt.bar(final_results["Model"], final_results["Accuracy"])

plt.title("Bradycardia Detection Model Comparison")
plt.xlabel("Models")
plt.ylabel("Accuracy")

plt.xticks(rotation=45)

plt.show()